In [2]:
import pandas as pd

df = pd.read_csv("../data/reviews_preprocessed.csv")

# Drop any rows where clean_text is empty
df = df.dropna(subset=['clean_text'])
df['clean_text'] = df['clean_text'].astype(str)

print("Dataset shape:", df.shape)
print("Label distribution:\n", df['label'].value_counts())

Dataset shape: (22496, 4)
Label distribution:
 label
0    11248
1    11248
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X = df['clean_text']   # Input: review text
y = df['label']        # Output: 0=genuine, 1=fake

# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 17996
Testing samples : 4500


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text to numbers
# max_features=10000 means we keep only top 10,000 most important words
vectorizer = TfidfVectorizer(max_features=10000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print("Training matrix shape:", X_train_tfidf.shape)
print("Each review is now a row of", X_train_tfidf.shape[1], "numbers")


Training matrix shape: (17996, 10000)
Each review is now a row of 10000 numbers


In [5]:
from sklearn.linear_model import LogisticRegression
import time

print("Training model... ")
start = time.time()

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

print(f"Done! Took {round(time.time()-start, 1)} seconds")

Training model... 
Done! Took 0.2 seconds


In [6]:
from sklearn.metrics import (accuracy_score, classification_report, 
                              confusion_matrix)

y_pred = model.predict(X_test_tfidf)

print("ACCURACY:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nDETAILED REPORT:")
print(classification_report(y_test, y_pred, 
      target_names=['Genuine', 'Fake']))

ACCURACY: 83.33 %

DETAILED REPORT:
              precision    recall  f1-score   support

     Genuine       0.85      0.80      0.83      2232
        Fake       0.82      0.86      0.84      2268

    accuracy                           0.83      4500
   macro avg       0.83      0.83      0.83      4500
weighted avg       0.83      0.83      0.83      4500



In [7]:
def predict_review(text):
    import re
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    
    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()
    
    text_clean = text.lower()
    text_clean = re.sub(r'[^a-z\s]', '', text_clean)
    words = text_clean.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    clean = ' '.join(words)
    
    vec = vectorizer.transform([clean])
    prediction = model.predict(vec)[0]
    probability = model.predict_proba(vec)[0]
    
    label = "🚨 FAKE" if prediction == 1 else "✅ GENUINE"
    confidence = round(max(probability) * 100, 1)
    
    print(f"Review    : {text[:80]}")
    print(f"Result    : {label}")
    print(f"Confidence: {confidence}%")
    print("-" * 50)

predict_review("Amazing food, best biryani in the city, must visit!")
predict_review("Good")
predict_review("Worst place ever do not go here terrible food and rude staff never coming back")
predict_review("I visited this restaurant last week with my family. The butter chicken was rich and creamy, service was warm and attentive. Will definitely return!")

Review    : Amazing food, best biryani in the city, must visit!
Result    : 🚨 FAKE
Confidence: 96.8%
--------------------------------------------------
Review    : Good
Result    : 🚨 FAKE
Confidence: 86.6%
--------------------------------------------------
Review    : Worst place ever do not go here terrible food and rude staff never coming back
Result    : 🚨 FAKE
Confidence: 95.9%
--------------------------------------------------
Review    : I visited this restaurant last week with my family. The butter chicken was rich 
Result    : 🚨 FAKE
Confidence: 75.2%
--------------------------------------------------


In [9]:
from sklearn.linear_model import LogisticRegression

# class_weight='balanced' forces model to treat both classes equally
model_balanced = LogisticRegression(
    max_iter=1000, 
    random_state=42,
    class_weight='balanced'   # ← this is the fix
)
model_balanced.fit(X_train_tfidf, y_train)

y_pred_b = model_balanced.predict(X_test_tfidf)

print("BALANCED MODEL ACCURACY:", 
      round(accuracy_score(y_test, y_pred_b) * 100, 2), "%")
print("\nDETAILED REPORT:")
print(classification_report(y_test, y_pred_b,
      target_names=['Genuine', 'Fake']))

BALANCED MODEL ACCURACY: 83.38 %

DETAILED REPORT:
              precision    recall  f1-score   support

     Genuine       0.85      0.80      0.83      2232
        Fake       0.82      0.86      0.84      2268

    accuracy                           0.83      4500
   macro avg       0.83      0.83      0.83      4500
weighted avg       0.83      0.83      0.83      4500



In [10]:
def predict_review_v2(text):
    import re
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    
    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()
    
    text_clean = text.lower()
    text_clean = re.sub(r'[^a-z\s]', '', text_clean)
    words = text_clean.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    clean = ' '.join(words)
    
    vec = vectorizer.transform([clean])
    prediction = model_balanced.predict(vec)[0]
    probability = model_balanced.predict_proba(vec)[0]
    
    label = "🚨 FAKE" if prediction == 1 else "✅ GENUINE"
    confidence = round(max(probability) * 100, 1)
    
    print(f"Review    : {text[:80]}")
    print(f"Result    : {label}")
    print(f"Confidence: {confidence}%")
    print("-" * 50)

predict_review_v2("Amazing food, best biryani in the city, must visit!")
predict_review_v2("Good")
predict_review_v2("Worst place ever do not go here terrible food and rude staff never coming back")
predict_review_v2("I visited this restaurant last week with my family. The butter chicken was rich and creamy, service was warm and attentive. Will definitely return!")

Review    : Amazing food, best biryani in the city, must visit!
Result    : 🚨 FAKE
Confidence: 96.9%
--------------------------------------------------
Review    : Good
Result    : 🚨 FAKE
Confidence: 86.2%
--------------------------------------------------
Review    : Worst place ever do not go here terrible food and rude staff never coming back
Result    : 🚨 FAKE
Confidence: 96.0%
--------------------------------------------------
Review    : I visited this restaurant last week with my family. The butter chicken was rich 
Result    : 🚨 FAKE
Confidence: 75.3%
--------------------------------------------------


In [11]:
import joblib

joblib.dump(model, '../models/fake_review_model.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')

print("Model saved to models/fake_review_model.pkl")
print("Vectorizer saved to models/tfidf_vectorizer.pkl")

Model saved to models/fake_review_model.pkl
Vectorizer saved to models/tfidf_vectorizer.pkl
